In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd


In [2]:
from datasets import load_dataset
dataset = load_dataset("fancyzhx/ag_news")
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [3]:
class_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
examples = []
for label_id, class_name in class_names.items():
    subset = train_dataset.filter(lambda x: x['label'] == label_id).select(range(3))
    for item in subset:
        examples.append({
            "Label ID": label_id,
            "Category": class_name,
            "Text": item["text"]
        })

In [4]:
df = pd.DataFrame(examples)
df

,Label ID,Category,Text
0,0,World,Venezuelans Vote Early in Referendum on Chavez...
1,0,World,S.Koreans Clash with Police on Iraq Troop Disp...
2,0,World,Palestinians in Israeli Jails Start Hunger Str...
3,1,Sports,"Phelps, Thorpe Advance in 200 Freestyle (AP) A..."
4,1,Sports,Reds Knock Padres Out of Wild-Card Lead (AP) A...
5,1,Sports,"Dreaming done, NBA stars awaken to harsh Olymp..."
6,2,Business,Wall St. Bears Claw Back Into the Black (Reute...
7,2,Business,Carlyle Looks Toward Commercial Aerospace (Reu...
8,2,Business,Oil and Economy Cloud Stocks' Outlook (Reuters...
9,3,Sci/Tech,"'Madden,' 'ESPN' Football Score in Different W..."


In [5]:
len(train_dataset) , len(test_dataset)

(120000, 7600)

In [6]:
from torch.utils.data import random_split
train , val = random_split(train_dataset,[0.9,0.1],generator=torch.Generator().manual_seed(42))

In [7]:
import spacy 
tokenizer = spacy.load("en_core_web_sm",disable = ["tagger", "parser", "ner", "lemmatizer", "attribute_ruler"])
def tokenize(text):
    return [token.text for token in tokenizer(text)]
example_text = "Wall St. Bears Claw Back Into the Black (Reuters)"
tokens = tokenize(example_text)
print(tokens)

['Wall', 'St.', 'Bears', 'Claw', 'Back', 'Into', 'the', 'Black', '(', 'Reuters', ')']


In [8]:
from collections import Counter
word_counter = Counter()
train_text = (item["text"] for item in train)
for doc in tokenizer.pipe(train_text,batch_size = 2000):
    token = [token.text.lower() for token in doc]
    word_counter.update(token)

special_token = ["<UNK>","<PAD>","<BOS>","<EOS>"]
word2index = {token : index for index , token in enumerate(special_token)}
min_freq = 1
for word ,count in word_counter.items():
    if count > min_freq:
        word2index[word] = len(word2index)
idex2word = {idx: word for word, idx in word2index.items()}


In [13]:
from gensim.models import KeyedVectors
model = KeyedVectors.load_word2vec_format(f"D:\Learn_NLP\LSTM\wiki-news-300d-1M.vec",binary = False)


<>:2: SyntaxWarning: invalid escape sequence '\L'
<>:2: SyntaxWarning: invalid escape sequence '\L'
C:\Users\MF\AppData\Local\Temp\ipykernel_1544\3123579532.py:2: SyntaxWarning: invalid escape sequence '\L'
  model = KeyedVectors.load_word2vec_format(f"D:\Learn_NLP\LSTM\wiki-news-300d-1M.vec",binary = False)


In [14]:
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create the embedding matrix for our vocabulary
vocab_size = len(word2index)
embedding_dim = model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in word2index.items():
    if word in model:
        embedding_matrix[idx] = model[word]
    else:
        # Initialize out-of-vocabulary or special tokens (except <PAD>) randomly
        if word == "<PAD>":
            embedding_matrix[idx] = np.zeros(embedding_dim)
        else:
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

# Convert to PyTorch tensor and create embedding layer
embedding_weights = torch.tensor(embedding_matrix, dtype=torch.float32)
embedding_layer = nn.Embedding.from_pretrained(embedding_weights, freeze=True)
embedding_layer = embedding_layer.to(device)


In [15]:
# Corrected the spelling to text_pipeline and fixed syntax/lookup logic
text_pipeline = lambda x: [word2index.get(token.lower(), word2index["<UNK>"]) for token in tokenize(x)]
# Corrected: Hugging Face datasets are already 0-indexed, so we don't subtract 1
label_pipeline = lambda x: int(x)
text_pipeline("Venezuelans Vote Early in Referendum on Chavez")


[27668, 75, 1173, 100, 5456, 122, 4433]

In [16]:
from torch.utils.data import DataLoader  # Fixed: torch.utils.data instead of torch.data.utils
from torch.nn.utils.rnn import pad_sequence

pad_index = word2index["<PAD>"]

def collect_fn(batch):
    label_list, text_list, length_list = [], [], []
    for item in batch:  # Fixed: Iterate items to access dictionary keys
        label_list.append(label_pipeline(item["label"]))
        
        # Fixed: Use torch.tensor instead of torch.Tensor, and fixed typo troch.int64 -> torch.int64
        process_text = torch.tensor(text_pipeline(item["text"]), dtype=torch.int64)
        text_list.append(process_text)
        length_list.append(process_text.size(0))
        
    # Fixed: Use torch.tensor instead of torch.Tensor for length_list
    return (
        torch.tensor(label_list, dtype=torch.int64),
        pad_sequence(text_list, padding_value=pad_index, batch_first=True),  # Fixed: padding_value instead of pad_value
        torch.tensor(length_list, dtype=torch.int64)
    )


In [17]:
batch_size = 64
train_loader = DataLoader(train,batch_size= batch_size,shuffle = True,collate_fn=collect_fn)
val_loader = DataLoader(val,batch_size= 500,shuffle=False,collate_fn=collect_fn)
test_loader = DataLoader(test_dataset,batch_size=500,shuffle=False,collate_fn=collect_fn)

In [18]:
import torch
import torch.nn as nn
class LSTM(nn.Module):
    def __init__(self,vocab_size,embedding_dim,pad_idx,hid_dim,num_layers,bidirectional,dropout,num_output):
        super(LSTM,self).__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim,padding_idx=pad_idx)
        self.lstm = nn.LSTM(embedding_dim,hid_dim,num_layers=num_layers,bidirectional=bidirectional,dropout=dropout,batch_first=True)
        self.fc = nn.Linear(hid_dim * 2,num_output)
    def forward(self,text,text_length):
        embedd = self.embedding(text)
        pack_embedd = nn.utils.rnn.pack_padded_sequence(embedd,text_length.to("cpu"),enforce_sorted=False,batch_first=True)

        packed_output , (hn,cn) = self.lstm(pack_embedd)
        output , output_length = nn.utils.rnn.pad_packed_sequence(packed_output,batch_first = True)
        hn = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim = 1)
        return self.fc(hn)

In [19]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, parm in m.named_parameters():
            if "bias" in name:
                nn.init.zeros_(parm)
            elif "weight" in name:
                nn.init.orthogonal_(parm)


In [20]:
input_dim = vocab_size
embedding_dim = 300
hid_dim = 256
num_output = 4
num_layers = 2
bidirectional = True
dropout = 0.5
lstm = LSTM(vocab_size,embedding_dim,pad_index,hid_dim,num_layers,bidirectional,dropout,num_output)
lstm.apply(init_weights)
lstm.embedding.weight.data.copy_(embedding_weights)


tensor([[ 0.9286, -0.4360, -0.2809,  ...,  0.0221,  1.1820, -0.2825],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [-0.8488, -0.9901, -1.2112,  ..., -0.3176, -0.6607,  0.4194],
        ...,
        [ 0.2040, -0.0477,  0.0104,  ...,  0.1839, -0.1099, -0.1225],
        [-1.7896,  0.9693,  0.1945,  ..., -0.7946, -1.0139, -0.0503],
        [-0.0733,  0.0246,  0.0449,  ..., -0.0796, -0.0786,  0.0936]])

In [21]:
def count_parm(model):
    parms = [p.numel() for p in model.parameters() if p.requires_grad]
    return sum(parms)
count_parm(lstm)

16812496

In [22]:
import torch.optim as optim
lr = 1e-3
optimizer = optim.Adam(lstm.parameters(),lr = lr)
criterion = nn.CrossEntropyLoss()


In [23]:
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report

# Move model to device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm = lstm.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm.parameters(), lr=1e-3)

In [24]:
def train_epoch(model, iterator, optimizer, criterion):
    epoch_loss = 0
    epoch_acc = 0
    
    model.train()
    
    for labels, texts, lengths in iterator:
        labels = labels.to(device)
        texts = texts.to(device)
        
        optimizer.zero_grad()
        
        predictions = model(texts, lengths)
        
        loss = criterion(predictions, labels)
        
        # Calculate accuracy
        preds = predictions.argmax(dim=1)
        acc = (preds == labels).sum().item() / len(labels)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_acc += acc
        
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion):
    epoch_loss = 0
    epoch_acc = 0
    
    all_preds = []
    all_labels = []
    
    model.eval()
    
    with torch.no_grad():
        for labels, texts, lengths in iterator:
            labels = labels.to(device)
            texts = texts.to(device)
            
            predictions = model(texts, lengths)
            loss = criterion(predictions, labels)
            
            preds = predictions.argmax(dim=1)
            acc = (preds == labels).sum().item() / len(labels)
            
            epoch_loss += loss.item()
            epoch_acc += acc
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Calculate macro F1 score
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    return epoch_loss / len(iterator), epoch_acc / len(iterator), f1, all_labels, all_preds

In [25]:
N_EPOCHS = 3

print("Starting training...")
for epoch in range(N_EPOCHS):
    train_loss, train_acc = train_epoch(lstm, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1, _, _ = evaluate(lstm, val_loader, criterion)
    
    print(f'Epoch: {epoch+1:02}')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {val_loss:.3f} |  Val. Acc: {val_acc*100:.2f}% | Val. F1: {val_f1:.4f}')

# Final Testing
print("\nEvaluating on Test Set...")    
test_loss, test_acc, test_f1, test_labels, test_preds = evaluate(lstm, test_loader, criterion)
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% | Test F1 (Macro): {test_f1:.4f}')

print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=list(class_names.values())))

Starting training...
Epoch: 01
	Train Loss: 0.302 | Train Acc: 89.65%
	 Val. Loss: 0.239 |  Val. Acc: 91.79% | Val. F1: 0.9176
Epoch: 02
	Train Loss: 0.171 | Train Acc: 94.28%
	 Val. Loss: 0.222 |  Val. Acc: 92.22% | Val. F1: 0.9222
Epoch: 03
	Train Loss: 0.115 | Train Acc: 96.08%
	 Val. Loss: 0.248 |  Val. Acc: 92.03% | Val. F1: 0.9202

Evaluating on Test Set...
Test Loss: 0.236 | Test Acc: 92.71% | Test F1 (Macro): 0.9242

Classification Report:
              precision    recall  f1-score   support

       World       0.95      0.91      0.93      1900
      Sports       0.96      0.99      0.97      1900
    Business       0.90      0.89      0.90      1900
    Sci/Tech       0.89      0.91      0.90      1900

    accuracy                           0.92      7600
   macro avg       0.92      0.92      0.92      7600
weighted avg       0.92      0.92      0.92      7600



In [26]:
torch.save(lstm.state_dict(),"LSTM.pth")